# Checkout A/B test: design and analysis

Follow-up to my theLook e-commerce analytics project. That analysis found 58% of carts
are abandoned and recommended fixing checkout first. This notebook shows how I would
test a new checkout page before rolling it out.

The test data is simulated. theLook has no experiment data, so I generated results
with the old checkout converting at 41.9% (the real rate from the project) and the new
one at 45% (the target). The notebook demonstrates the method, not a real result.


In [1]:
from statsmodels.stats.proportion import proportion_effectsize
from statsmodels.stats.power import NormalIndPower

baseline = 0.419   # cart-to-purchase rate today
target = 0.45      # the rate we hope the new checkout reaches
alpha = 0.05       # we accept a 5% risk of a false alarm
power = 0.80       # we want an 80% chance of catching a real improvement

effect = proportion_effectsize(target, baseline)
n_per_group = NormalIndPower().solve_power(
    effect_size=effect, alpha=alpha, power=power, ratio=1, alternative='two-sided'
)
print(round(n_per_group))

4012


In [2]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 4012

df = pd.DataFrame({
    'group': ['A'] * n + ['B'] * n,
    'purchased': np.concatenate([rng.binomial(1, 0.419, n), rng.binomial(1, 0.45, n)])
})
summary = df.groupby('group')['purchased'].agg(carts='count', purchases='sum', rate='mean')
print(summary)

       carts  purchases      rate
group                            
A       4012       1664  0.414756
B       4012       1780  0.443669


In [3]:
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

a = summary.loc['A']; b = summary.loc['B']
z, p = proportions_ztest([b.purchases, a.purchases], [b.carts, a.carts])
low, high = confint_proportions_2indep(b.purchases, b.carts, a.purchases, a.carts, method='wald')
print('p-value:', round(p, 4))
print('lift in points:', round((b.rate - a.rate) * 100, 2))
print('95% interval in points:', round(low * 100, 2), 'to', round(high * 100, 2))

p-value: 0.0089
lift in points: 2.89
95% interval in points: 0.73 to 5.06


## Decision

- Hypothesis: the new checkout raises the cart-to-purchase rate.
- Design: 4,012 carts per group, giving 80% power to detect a rise from 41.9% to 45% at 5% significance.
- Result: A converted 41.48% (1,664 of 4,012) and B converted 44.37% (1,780 of 4,012). The lift is 2.89 points, p = 0.0089, 95% interval 0.73 to 5.06 points.
- Decision: roll out the new checkout. The difference is statistically significant.
- Value: on the historical 431,636 carts, 2.89 points is about 12,500 extra purchases, worth roughly 560,000 at 44.84 per purchase. The interval puts the range at about 140,000 to 980,000.
- Caveats: the interval is wide, so the true lift could be under 1 point. After rollout I would keep tracking the rate, and check that the cancel-or-return rate did not rise.